<a href="https://colab.research.google.com/github/HugoCortazar/Mineria-de-Textos-/blob/main/MTPracticaFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica de Minería de Textos: Clasificación de textos en Esperanto

## 1. Introducción

El objetivo de esta práctica es desarrollar un sistema de minería de textos capaz de clasificar textos escritos en Esperanto según su temática. Para ello, se trabajará con tres categorías científicas: **biología (`biologio`), química (`kemio`) y física (`fiziko`)**.

El proyecto se divide en varias fases que permiten construir progresivamente el sistema. En esta primera parte se desarrollarán las dos fases iniciales: la obtención y el preprocesamiento del conjunto de datos (*dataset*), y la implementación de un modelo propio de procesamiento del lenguaje natural (*NLP*).

En la **Fase 1**, se utilizará la API de Wikipedia en Esperanto para obtener automáticamente artículos relacionados con las tres categorías seleccionadas. El contenido obtenido se dividirá en párrafos y se someterá a un proceso de limpieza para eliminar referencias, enlaces y otros elementos que puedan introducir ruido. Finalmente, se construirá un dataset etiquetado con dos atributos: `text`, que contiene el texto de cada párrafo, y `class`, que identifica su categoría temática. También se analizará la cantidad de información obtenida y el efecto de las reglas de limpieza aplicadas.

En la **Fase 2**, se desarrollará desde cero un modelo de NLP basado en reglas, aprovechando la regularidad gramatical del Esperanto. El modelo incluirá un tokenizador, un lematizador, un etiquetador morfológico y un mecanismo de gestión de *stop-words*. Para organizar la información lingüística se implementarán las clases `Token` y `Doc`, mediante una interfaz inspirada en spaCy, pero sin utilizar bibliotecas de NLP preexistentes.

El desarrollo se realizará en Python mediante un cuaderno Jupyter, utilizando herramientas como Pandas para la gestión del dataset y las bibliotecas estándar de Python para la obtención y el procesamiento del texto.

A lo largo de estas fases se prestará especial atención a la reproducibilidad del proceso, la calidad de los datos y la comprobación del funcionamiento de las reglas lingüísticas. Asimismo, se documentarán las decisiones de diseño y las limitaciones encontradas, con el objetivo de disponer de una base sólida para las siguientes fases de la práctica, en las que se abordará la clasificación automática de textos.

In [1]:
# ==============================================================================
# CELDA 1. IMPORTS, PARAMETROS Y ARTICULOS SEMILLA
# ==============================================================================

import json
import re
import time
import urllib.parse
import urllib.request

from urllib.error import HTTPError, URLError
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

import pandas as pd

# True: generar un CSV nuevo desde Wikipedia.
# False: cargar el CSV local o desde GitHub.
DESCARGAR_DESDE_WIKIPEDIA = True

GITHUB_CSV_URL = (
    "https://raw.githubusercontent.com/"
    "HugoCortazar/Mineria-de-Textos-/main/"
    "esperanto_scienco_dataset.csv"
)

WIKIPEDIA_API_URL = "https://eo.wikipedia.org/w/api.php"

USER_AGENT = (
    "MineriaTextosEsperanto/1.0 "
    "(https://github.com/HugoCortazar/Mineria-de-Textos-/)"
)

LONGITUD_MINIMA_PARRAFO = 60
PAUSA_ENTRE_ARTICULOS = 2

# Control interno para impedir guardar datasets claramente incompletos.
# No es un mínimo oficial de la asignatura.
MIN_PARRAFOS_POR_CLASE = 30

CATEGORIAS = {
    "biologio": [
        "Biologio", "Ĉelo", "Genetiko", "Evoluismo",
        "Fotosintezo", "Zoologio", "Botaniko",
        "Mikrobiologio", "Ekologio", "DNA"
    ],
    "kemio": [
        "Kemio", "Atomo", "Perioda tabelo", "Kemia ligo",
        "Organika kemio", "Neorganika kemio",
        "Kemia reakcio", "Acido", "Molekulo", "Katalizilo"
    ],
    "fiziko": [
        "Fiziko", "Klasika mekaniko", "Termodinamiko",
        "Elektromagnetismo", "Kvantuma mekaniko",
        "Relativeco", "Optiko", "Kinetika energio",
        "Partikla fiziko", "Astrofiziko"
    ]
}

SECCIONES_RUIDOSAS = {
    "referencoj",
    "referencoj kaj notoj",
    "eksteraj ligiloj",
    "vidu ankaŭ",
    "vidu ankau",
    "bibliografio",
    "literaturo",
    "notoj",
    "fontoj"
}

print("Configuración cargada.")
print("Descarga desde Wikipedia:", DESCARGAR_DESDE_WIKIPEDIA)
print("Categorías:", list(CATEGORIAS.keys()))

Configuración cargada.
Descarga desde Wikipedia: True
Categorías: ['biologio', 'kemio', 'fiziko']


In [2]:
# ==============================================================================
# CELDA 2. FUNCIONES DE DESCARGA DE WIKIPEDIA
# ==============================================================================

def calcular_espera_reintento(cabeceras, intento):
    """Respeta Retry-After; si falta, utiliza una pausa creciente."""

    espera_base = 10 * (2 ** intento)
    retry_after = cabeceras.get("Retry-After") if cabeceras else None

    if not retry_after:
        return espera_base

    try:
        return max(espera_base, float(retry_after))
    except (TypeError, ValueError):
        try:
            fecha = parsedate_to_datetime(retry_after)

            if fecha.tzinfo is None:
                fecha = fecha.replace(tzinfo=timezone.utc)

            segundos = (
                fecha - datetime.now(timezone.utc)
            ).total_seconds()

            return max(espera_base, segundos, 0)

        except (TypeError, ValueError, OverflowError):
            return espera_base


def obtener_texto_wikipedia(titulo, max_intentos=4):
    """
    Devuelve el texto plano de un único artículo de Wikipedia en Esperanto.
    """

    parametros = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "redirects": 1,
        "titles": titulo,
        "format": "json",
        "formatversion": 2
    }

    url = (
        WIKIPEDIA_API_URL
        + "?"
        + urllib.parse.urlencode(parametros)
    )

    for intento in range(max_intentos):

        solicitud = urllib.request.Request(
            url,
            headers={
                "User-Agent": USER_AGENT,
                "Accept": "application/json"
            }
        )

        try:
            with urllib.request.urlopen(
                solicitud, timeout=30
            ) as respuesta:

                datos = json.loads(
                    respuesta.read().decode("utf-8")
                )

            paginas = datos.get("query", {}).get("pages", [])

            if not paginas:
                print(
                    f"Aviso: la API no devolvió páginas para '{titulo}'."
                )
                return ""

            pagina = paginas[0]

            if pagina.get("missing"):
                print(
                    f"Aviso: no existe el artículo '{titulo}' "
                    "en Wikipedia Esperanto."
                )
                return ""

            texto = pagina.get("extract", "")

            if not texto.strip():
                print(
                    f"Aviso: el artículo '{titulo}' "
                    "no contiene extracto de texto."
                )
                return ""

            return texto

        except HTTPError as error:

            if error.code not in (429, 503):
                print(
                    f"Aviso HTTP {error.code} para '{titulo}': "
                    f"{error.reason}"
                )
                return ""

            if intento == max_intentos - 1:
                raise RuntimeError(
                    f"Wikipedia continúa limitando las peticiones "
                    f"para '{titulo}' tras {max_intentos} intentos. "
                    "Espera antes de volver a ejecutar."
                ) from error

            espera = calcular_espera_reintento(
                error.headers, intento
            )

            # No dejar Colab bloqueado durante esperas extraordinariamente largas.
            if espera > 300:
                raise RuntimeError(
                    f"Wikipedia indica que hay que esperar "
                    f"{espera:.0f} segundos antes de reintentar "
                    f"'{titulo}'. Ejecuta el cuaderno más tarde."
                ) from error

            print(
                f"HTTP {error.code} al consultar '{titulo}'. "
                f"Esperando {espera:.0f} segundos antes de reintentar."
            )

            time.sleep(espera)

        except (URLError, TimeoutError) as error:

            if intento == max_intentos - 1:
                raise RuntimeError(
                    f"No se pudo conectar con Wikipedia para "
                    f"'{titulo}' después de {max_intentos} intentos."
                ) from error

            espera = 5 * (2 ** intento)

            print(
                f"Error de conexión para '{titulo}'. "
                f"Reintento en {espera} segundos."
            )

            time.sleep(espera)

    return ""


print(
    "Función definida: obtener_texto_wikipedia(titulo), "
    "en singular."
)

Función definida: obtener_texto_wikipedia(titulo), en singular.


In [3]:
# ==============================================================================
# CELDA 3. LIMPIEZA Y PREPROCESAMIENTO DEL TEXTO
# ==============================================================================

def limpiar_y_filtrar_parrafos(texto_articulo, clase):
    """
    Devuelve:
    - Párrafos válidos etiquetados.
    - Número de bloques considerados.
    - Número de bloques descartados.
    """

    parrafos_validos = []
    total_bloques = 0
    ruido_detectado = 0
    omitir_seccion = False

    for linea in texto_articulo.splitlines():

        p = linea.strip()

        if not p:
            continue

        total_bloques += 1
        titulo_seccion = None

        # Detectar títulos con formato Wiki.
        encabezado = re.match(
            r"^(={2,})\s*(.*?)\s*\1\s*$",
            p
        )

        if encabezado:
            titulo_seccion = encabezado.group(2).strip().lower()

        elif p.lower().strip(" .:") in SECCIONES_RUIDOSAS:
            # Algunos encabezados pueden aparecer como texto plano.
            titulo_seccion = p.lower().strip(" .:")

        if titulo_seccion is not None:

            omitir_seccion = any(
                nombre in titulo_seccion
                for nombre in SECCIONES_RUIDOSAS
            )

            ruido_detectado += 1
            continue

        if omitir_seccion:
            ruido_detectado += 1
            continue

        # Eliminar referencias, URLs y posibles etiquetas HTML.
        p = re.sub(r"\[\d+\]", "", p)
        p = re.sub(r"https?://\S+", "", p)
        p = re.sub(r"<[^>]+>", " ", p)

        # El CSV utiliza ; como separador.
        p = p.replace(";", ",")

        p = re.sub(r"\s+", " ", p).strip()

        # Filtro por longitud.
        if len(p) < LONGITUD_MINIMA_PARRAFO:
            ruido_detectado += 1
            continue

        parrafos_validos.append((p, clase))

    return parrafos_validos, total_bloques, ruido_detectado


print(
    "Función definida: "
    "limpiar_y_filtrar_parrafos(texto_articulo, clase)."
)

Función definida: limpiar_y_filtrar_parrafos(texto_articulo, clase).


In [4]:
# ==============================================================================
# CELDA 4. VALIDACION, GENERACION O CARGA DEL DATASET
# ==============================================================================

def validar_dataset(df):
    """Valida el contrato text;class y limpia filas problemáticas."""

    columnas = ["text", "class"]

    faltantes = [
        col for col in columnas
        if col not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Faltan columnas obligatorias: {faltantes}"
        )

    df = df[columnas].copy()
    df = df.dropna(subset=columnas)

    df["text"] = df["text"].astype(str).str.strip()
    df["class"] = df["class"].astype(str).str.strip()

    df = df[
        (df["text"] != "") &
        (df["class"] != "")
    ]

    df["text"] = df["text"].str.replace(
        ";", ",", regex=False
    )

    # Eliminar duplicados exactos con la misma etiqueta.
    df = df.drop_duplicates(subset=["text", "class"])

    # Eliminar textos idénticos que tengan etiquetas contradictorias.
    clases_por_texto = df.groupby("text")["class"].nunique()

    textos_ambiguos = clases_por_texto[
        clases_por_texto > 1
    ].index

    if len(textos_ambiguos) > 0:
        print(
            "Textos con etiquetas contradictorias descartados:",
            len(textos_ambiguos)
        )

        df = df[
            ~df["text"].isin(textos_ambiguos)
        ]

    return df.reset_index(drop=True)


def generar_dataset():
    """Descarga, limpia y construye el dataset de las tres categorías."""

    filas = []
    bloques_totales = 0
    bloques_ruidosos = 0

    articulos_con_texto = {
        clase: 0 for clase in CATEGORIAS
    }

    for clase, articulos in CATEGORIAS.items():

        print(
            f"\nDescargando artículos de la categoría: {clase}"
        )

        parrafos_clase = 0

        for articulo in articulos:

            # IMPORTANTE: función singular, una petición por artículo.
            texto = obtener_texto_wikipedia(articulo)

            if texto:

                articulos_con_texto[clase] += 1

                parrafos, total, ruido = (
                    limpiar_y_filtrar_parrafos(
                        texto, clase
                    )
                )

                filas.extend(parrafos)

                bloques_totales += total
                bloques_ruidosos += ruido

                parrafos_clase += len(parrafos)

            # Espaciar las peticiones a Wikipedia.
            time.sleep(PAUSA_ENTRE_ARTICULOS)

        print(
            f"Artículos con texto: "
            f"{articulos_con_texto[clase]} de {len(articulos)}"
        )

        print(
            f"Párrafos válidos en {clase}: "
            f"{parrafos_clase}"
        )

    df_resultado = validar_dataset(
        pd.DataFrame(filas, columns=["text", "class"])
    )

    conteos = df_resultado["class"].value_counts()

    print("\n--- MÉTRICAS DE LIMPIEZA ---")
    print("Bloques no vacíos considerados:", bloques_totales)
    print("Bloques descartados por las reglas:", bloques_ruidosos)

    tasa = (
        bloques_ruidosos / bloques_totales * 100
        if bloques_totales else 0
    )

    print(f"Tasa estimada de descarte: {tasa:.2f}%")

    print("\n--- PÁRRAFOS POR CLASE ---")

    for clase in CATEGORIAS:
        print(
            f"{clase}: {int(conteos.get(clase, 0))}"
        )

    clases_vacias = [
        clase for clase in CATEGORIAS
        if conteos.get(clase, 0) == 0
    ]

    clases_insuficientes = [
        clase for clase in CATEGORIAS
        if conteos.get(clase, 0) < MIN_PARRAFOS_POR_CLASE
    ]

    if df_resultado.empty or clases_vacias:
        raise ValueError(
            "Dataset incompleto. Clases sin texto: "
            f"{clases_vacias}. No se guardará el CSV."
        )

    if clases_insuficientes:
        raise ValueError(
            f"Hay clases con menos de {MIN_PARRAFOS_POR_CLASE} "
            f"párrafos: {clases_insuficientes}. "
            "Amplía los artículos semilla o comprueba la descarga. "
            "No se ha guardado el CSV."
        )

    # Guardar únicamente si se han superado los controles.
    df_resultado.to_csv(
        "esperanto_scienco_dataset.csv",
        sep=";",
        index=False,
        encoding="utf-8"
    )

    print("\nDataset guardado: esperanto_scienco_dataset.csv")
    print("Filas totales:", len(df_resultado))
    print(df_resultado.head())

    print(
        "\nNota: el enunciado recomienda más de 300 "
        "párrafos por clase."
    )

    if any(
        conteos.get(clase, 0) < 300
        for clase in CATEGORIAS
    ):
        print(
            "Aviso: alguna clase sigue por debajo "
            "de la recomendación de 300 párrafos."
        )

    return df_resultado


# Ejecutar la generación o cargar un dataset existente.
if DESCARGAR_DESDE_WIKIPEDIA:

    df = generar_dataset()

else:

    try:
        df = pd.read_csv(
            "esperanto_scienco_dataset.csv",
            sep=";",
            encoding="utf-8"
        )

        print("Dataset cargado desde el archivo local.")

    except FileNotFoundError:

        print(
            "No hay CSV local; se intentará cargar desde GitHub."
        )

        try:
            df = pd.read_csv(
                GITHUB_CSV_URL,
                sep=";",
                encoding="utf-8"
            )

        except Exception as error:
            raise RuntimeError(
                "No se pudo cargar el CSV local ni desde GitHub. "
                "Genera el dataset o publica el CSV en el repositorio."
            ) from error

    df = validar_dataset(df)

    clases_encontradas = set(df["class"].unique())
    clases_esperadas = set(CATEGORIAS.keys())

    if not clases_esperadas.issubset(clases_encontradas):
        raise ValueError(
            "El CSV no contiene las tres clases. "
            f"Clases encontradas: {clases_encontradas}. "
            "No continúes con este dataset incompleto."
        )

    print("Filas cargadas:", len(df))
    print(df["class"].value_counts())


Descargando artículos de la categoría: biologio
Artículos con texto: 10 de 10
Párrafos válidos en biologio: 267

Descargando artículos de la categoría: kemio
Artículos con texto: 10 de 10
Párrafos válidos en kemio: 443

Descargando artículos de la categoría: fiziko
Artículos con texto: 10 de 10
Párrafos válidos en fiziko: 327

--- MÉTRICAS DE LIMPIEZA ---
Bloques no vacíos considerados: 3902
Bloques descartados por las reglas: 2865
Tasa estimada de descarte: 73.42%

--- PÁRRAFOS POR CLASE ---
biologio: 267
kemio: 439
fiziko: 325

Dataset guardado: esperanto_scienco_dataset.csv
Filas totales: 1031
                                                text     class
0  Biologio estas la scienco pri la fenomenoj de ...  biologio
1  Tiu scienco studas la karakterojn, nome fizika...  biologio
2  Biologio ampleksas larĝan spektron de sciencaj...  biologio
3  Sub-disciplinoj de biologio estas difinitaj la...  biologio
4  Je pli vasta skalo, genetiko studas kiel hered...  biologio

Nota: el enuncia

In [5]:
# ==============================================================================
# CELDA 5. STOP-WORDS Y PALABRAS FUNCIONALES
# ==============================================================================

ESPERANTO_STOPWORDS = {
    "la", "kaj", "de", "en", "al", "estas", "esti",
    "estis", "estos", "por", "kun", "per", "sur",
    "pri", "sed", "ĉar", "ke", "ne", "kiel", "tiu",
    "tiuj", "tiun", "tiujn", "kiu", "kiuj", "kiun",
    "kiujn", "mi", "vi", "li", "ŝi", "ĝi", "ili",
    "ni", "oni", "sia", "sian", "siaj", "siajn",
    "ankaŭ", "nur", "jam", "pli", "plej", "tre",
    "tro", "ĉe", "post", "dum"
}

# Palabras funcionales frecuentes.
# Las etiquetas POS son la convención utilizada en este proyecto.

PALABRAS_FUNCION = {
    "la": ("la", "DET"),

    "kaj": ("kaj", "CCONJ"),
    "sed": ("sed", "CCONJ"),
    "ĉar": ("ĉar", "SCONJ"),
    "ke": ("ke", "SCONJ"),

    "de": ("de", "ADP"),
    "en": ("en", "ADP"),
    "al": ("al", "ADP"),
    "por": ("por", "ADP"),
    "kun": ("kun", "ADP"),
    "per": ("per", "ADP"),
    "sur": ("sur", "ADP"),
    "pri": ("pri", "ADP"),
    "ĉe": ("ĉe", "ADP"),
    "post": ("post", "ADP"),
    "dum": ("dum", "ADP"),

    "ne": ("ne", "PART"),

    "kiel": ("kiel", "ADV"),
    "jam": ("jam", "ADV"),
    "ankaŭ": ("ankaŭ", "ADV"),
    "nur": ("nur", "ADV"),

    "mi": ("mi", "PRON"),
    "vi": ("vi", "PRON"),
    "li": ("li", "PRON"),
    "ŝi": ("ŝi", "PRON"),
    "ĝi": ("ĝi", "PRON"),
    "ili": ("ili", "PRON"),
    "ni": ("ni", "PRON"),
    "oni": ("oni", "PRON"),

    "tiu": ("tiu", "DET"),
    "tiuj": ("tiu", "DET"),
    "tiun": ("tiu", "DET"),
    "tiujn": ("tiu", "DET"),

    "kiu": ("kiu", "PRON"),
    "kiuj": ("kiu", "PRON"),
    "kiun": ("kiu", "PRON"),
    "kiujn": ("kiu", "PRON"),
}

print("Stop-words y palabras funcionales cargadas.")

Stop-words y palabras funcionales cargadas.


In [6]:
# ==============================================================================
# CELDA 6. OBJETOS TOKEN, DOC Y MODELO ESPERANTONLP
# ==============================================================================

class Token:
    """Representa un token con su lema y propiedades gramaticales."""

    def __init__(self, text, lemma, pos, is_stop, is_punct):
        self.text = text
        self.lemma_ = lemma
        self.pos_ = pos
        self.is_stop = is_stop
        self.is_punct = is_punct

    def __repr__(self):
        return (
            f"Token(text={self.text!r}, lemma={self.lemma_!r}, "
            f"pos={self.pos_!r}, stop={self.is_stop}, "
            f"punct={self.is_punct})"
        )


class Doc:
    """Contenedor iterable de tokens que conserva el texto original."""

    def __init__(self, tokens, original_text):
        self._tokens = tokens
        self._text = original_text

    def __iter__(self):
        return iter(self._tokens)

    def __len__(self):
        return len(self._tokens)

    def __getitem__(self, index):
        return self._tokens[index]

    @property
    def text(self):
        return self._text

    def lemmatized_text(
        self,
        remove_stopwords=True,
        remove_punct=True
    ):
        lemmas = []

        for token in self._tokens:

            if remove_punct and token.is_punct:
                continue

            if remove_stopwords and token.is_stop:
                continue

            lemmas.append(token.lemma_)

        return " ".join(lemmas)


class EsperantoNLP:
    """Pipeline aproximado basado en reglas para texto en Esperanto."""

    def __init__(self, stopwords=None):
        self.stopwords = (
            set(ESPERANTO_STOPWORDS)
            if stopwords is None
            else set(stopwords)
        )

    def _morphological_analysis(self, word):
        w = word.lower()

        # 1. Palabras funcionales conocidas.
        if w in PALABRAS_FUNCION:
            return PALABRAS_FUNCION[w]

        # 2. Verbos conjugados regulares.
        for ending in ("as", "is", "os", "us"):

            if w.endswith(ending) and len(w) > len(ending):
                return w[:-len(ending)] + "i", "VERB"

        if w.endswith("u") and len(w) > 2:
            return w[:-1] + "i", "VERB"

        if w.endswith("i") and len(w) > 2:
            return w, "VERB"

        # 3. Eliminación controlada de -j y -n.
        base = w

        if (
            len(w) > 3
            and w.endswith("jn")
            and w[:-2].endswith(("o", "a"))
        ):
            base = w[:-2]

        elif (
            len(w) > 2
            and w.endswith("n")
            and w[:-1].endswith(("o", "a"))
        ):
            base = w[:-1]

        elif (
            len(w) > 2
            and w.endswith("j")
            and w[:-1].endswith(("o", "a"))
        ):
            base = w[:-1]

        # 4. Terminaciones nominales y adjetivales.
        if base.endswith("o") and len(base) > 1:
            return base, "NOUN"

        if base.endswith("a") and len(base) > 1:
            return base, "ADJ"

        if base.endswith("e") and len(base) > 1:
            return base, "ADV"

        return w, "X"

    def __call__(self, text):

        if not isinstance(text, str):
            raise TypeError(
                "EsperantoNLP espera una cadena de texto."
            )

        # \w reconoce letras Unicode, incluidas las del Esperanto.
        raw_tokens = re.findall(
            r"\w+|[^\w\s]",
            text,
            flags=re.UNICODE
        )

        doc_tokens = []

        for raw in raw_tokens:

            is_punct = bool(
                re.fullmatch(
                    r"[^\w\s]",
                    raw,
                    flags=re.UNICODE
                )
            )

            lower_raw = raw.lower()

            if is_punct:
                lemma, pos = raw, "PUNCT"
            else:
                lemma, pos = self._morphological_analysis(raw)

            token = Token(
                text=raw,
                lemma=lemma,
                pos=pos,
                is_stop=lower_raw in self.stopwords,
                is_punct=is_punct
            )

            doc_tokens.append(token)

        return Doc(doc_tokens, text)


nlp = EsperantoNLP()

print("Modelo EsperantoNLP inicializado.")

Modelo EsperantoNLP inicializado.


In [7]:
# ==============================================================================
# CELDA 7. DEMOSTRACION Y PRUEBAS DEL MODELO NLP
# ==============================================================================

ejemplo = "La grandaj ĉeloj dividiĝas rapide en la laboratorio."

doc = nlp(ejemplo)

print("TEXTO ORIGINAL:")
print(doc.text)

print("\nTOKENS:")
print(
    f"{'TEXTO':<15} | {'LEMA':<15} | "
    f"{'POS':<8} | {'STOPWORD':<9} | PUNT"
)
print("-" * 65)

for token in doc:
    print(
        f"{token.text:<15} | {token.lemma_:<15} | "
        f"{token.pos_:<8} | {str(token.is_stop):<9} | "
        f"{token.is_punct}"
    )

print("\nTexto lematizado sin stop-words ni puntuación:")

print(
    doc.lemmatized_text(
        remove_stopwords=True,
        remove_punct=True
    )
)

# Pruebas básicas de regresión.
pruebas = {
    "kaj": ("kaj", "CCONJ", True),
    "kun": ("kun", "ADP", True),
    "en": ("en", "ADP", True),
    "ĉeloj": ("ĉelo", "NOUN", False),
    "ĉelojn": ("ĉelo", "NOUN", False),
    "grandajn": ("granda", "ADJ", False),
    "estas": ("esti", "VERB", True),
    "rapide": ("rapide", "ADV", False),
}

for palabra, esperado in pruebas.items():

    token = nlp(palabra)[0]

    obtenido = (
        token.lemma_,
        token.pos_,
        token.is_stop
    )

    assert obtenido == esperado, (
        f"Error con {palabra!r}: "
        f"esperado {esperado}, obtenido {obtenido}"
    )

# Pruebas del objeto Doc.
assert nlp(ejemplo).text == ejemplo
assert len(nlp("")) == 0

# Prueba de puntuación y lematización.
assert nlp("Saluton.").lemmatized_text() == "saluto"

print("\nTodas las pruebas básicas del modelo NLP han pasado.")

TEXTO ORIGINAL:
La grandaj ĉeloj dividiĝas rapide en la laboratorio.

TOKENS:
TEXTO           | LEMA            | POS      | STOPWORD  | PUNT
-----------------------------------------------------------------
La              | la              | DET      | True      | False
grandaj         | granda          | ADJ      | False     | False
ĉeloj           | ĉelo            | NOUN     | False     | False
dividiĝas       | dividiĝi        | VERB     | False     | False
rapide          | rapide          | ADV      | False     | False
en              | en              | ADP      | True      | False
la              | la              | DET      | True      | False
laboratorio     | laboratorio     | NOUN     | False     | False
.               | .               | PUNCT    | False     | True

Texto lematizado sin stop-words ni puntuación:
granda ĉelo dividiĝi rapide laboratorio

Todas las pruebas básicas del modelo NLP han pasado.
